#### Imports and load

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from math import radians, sin, cos, sqrt, atan2

# Load the data
train = pd.read_csv('../data/raw/fraudTrain.csv')
test = pd.read_csv('../data/raw/fraudTest.csv')

# Drop junk column
train = train.drop(columns=['Unnamed: 0'])
test = test.drop(columns=['Unnamed: 0'])

print("Data loaded successfully")
print(f"Train: {train.shape} | Test: {test.shape}")

Data loaded successfully
Train: (1296675, 22) | Test: (555719, 22)


In [7]:
print(train['trans_date_trans_time'])

0         2019-01-01 00:00:18
1         2019-01-01 00:00:44
2         2019-01-01 00:00:51
3         2019-01-01 00:01:16
4         2019-01-01 00:03:06
                  ...        
1296670   2020-06-21 12:12:08
1296671   2020-06-21 12:12:19
1296672   2020-06-21 12:12:32
1296673   2020-06-21 12:13:36
1296674   2020-06-21 12:13:37
Name: trans_date_trans_time, Length: 1296675, dtype: datetime64[us]


#### Time Features

In [6]:
# Convert transaction time from string to datetime object
train['trans_date_trans_time'] = pd.to_datetime(train['trans_date_trans_time'])
test['trans_date_trans_time'] = pd.to_datetime(test['trans_date_trans_time'])

# Extract meaningful time components from the training set
train['hour'] = train['trans_date_trans_time'].dt.hour
train['day_of_week'] = train['trans_date_trans_time'].dt.day_of_week
train['month'] = train['trans_date_trans_time'].dt.month
train['is_weekend'] = train['day_of_week'].isin([5, 6]).astype(int)

# Extract meaningful time components from the test set
test['hour'] = test['trans_date_trans_time'].dt.hour
test['day_of_week'] = test['trans_date_trans_time'].dt.day_of_week
test['month'] = test['trans_date_trans_time'].dt.month
test['is_weekend'] = test['day_of_week'].isin([5,6]).astype(int)

# Quick check - fraud by hour
fraud_by_hour = train.groupby('hour')['is_fraud'].mean() * 100
print("Fraud rate by hour:")
print(fraud_by_hour.round(2))

Fraud rate by hour:
hour
0     1.49
1     1.53
2     1.47
3     1.42
4     0.11
5     0.14
6     0.09
7     0.13
8     0.12
9     0.11
10    0.09
11    0.10
12    0.10
13    0.12
14    0.13
15    0.12
16    0.12
17    0.12
18    0.12
19    0.12
20    0.10
21    0.11
22    2.88
23    2.84
Name: is_fraud, dtype: float64


#### Age Feature

In [12]:
# Convert date of birth to datetime
train['dob'] = pd.to_datetime(train['dob'])
test['dob'] = pd.to_datetime(test['dob'])

# Calculate age at time of transaction
train['age'] = (train['trans_date_trans_time'] - train['dob']).dt.days // 365
test['age'] = (test['trans_date_trans_time'] - test['dob']).dt.days // 365

# Check age distribution by fraud class
print(train.groupby('is_fraud')['age'].describe())

              count       mean        std   min   25%   50%   75%   max
is_fraud                                                               
0         1289169.0  45.511960  17.398818  13.0  32.0  43.0  57.0  95.0
1            7506.0  48.321609  18.864543  14.0  33.0  47.0  60.0  93.0
